In [1]:
### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime
import h5py
import numpy as np

### Pick data for ENA from MODIS

In [6]:
from pyhdf.SD import SD, SDC

fn = '/data/ggong/MODIS/ENA/MAC06S0.A2017339.0400.002.2017350205953.hdf'
hdf = SD(fn, SDC.READ)

vars_to_check = [
    "Cloud_Effective_Radius_1621",
    "Cloud_Water_Path_1621",
    "Cloud_Phase_Optical_Properties",
    "cloud_top_temperature_1km",
    "Latitude",
    "Longitude"
]

for v in vars_to_check:
    sds = hdf.select(v)
    print(f"{v}: shape = {sds[:].shape}")

Cloud_Effective_Radius_1621: shape = (2040, 11)
Cloud_Water_Path_1621: shape = (2040, 11)
Cloud_Phase_Optical_Properties: shape = (2040, 11)
cloud_top_temperature_1km: shape = (2040, 11)
Latitude: shape = (408, 3)
Longitude: shape = (408, 3)


In [ ]:
import os
import glob
import numpy as np
import xarray as xr
from pyhdf.SD import SD, SDC

# ===== paths =====
IN_DIR = "/data/ggong/MODIS/ENA_5x5"
OUT_DIR = "/data/ggong/MODIS/ENA_nc_5x5"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== ENA box =====
ena_lat_min, ena_lat_max = 36.5916, 41.5916
ena_lon_min, ena_lon_max = -30.5257, -25.5257

def read_scaled_nan(hdf, name, out_dtype=np.float32):
    sds = hdf.select(name)
    arr = sds.get().astype(out_dtype)
    a = sds.attributes()
    fill = a.get("_FillValue", None)
    scale = float(a.get("scale_factor", 1.0))
    offset = float(a.get("add_offset", 0.0))
    if fill is not None:
        arr = np.where(arr == fill, np.nan, arr)
    arr = arr * scale + offset
    return arr

def secs93_to_dt64ms(sec):
    epoch = np.datetime64("1993-01-01T00:00:00")
    ms = np.round(sec * 1000.0).astype(np.int64)
    return epoch + ms.astype("timedelta64[ms]")

def agg_1km_to_5km(x_1km, lat_shape):
    """
    Aggregate 1 km variables from shape (2030, 11) to
    5 km geolocation shape (406, 3) using nanmean.

    This also handles the ENA subset special case:
    cross-track dimension 11 -> 3 using groups 3-5-3.
    """
    n5, m5 = lat_shape
    n1, m1 = x_1km.shape
    if n1 != n5 * 5:
        raise ValueError(f"along-track mismatch: {x_1km.shape} vs {lat_shape}")

    x_along5 = np.nanmean(x_1km.reshape(n5, 5, m1), axis=1)  # (n5, m1)

    # ENA subset special case: 11 -> 3 using 3-5-3 grouping
    if (m1, m5) == (11, 3):
        g = [(0, 3), (3, 8), (8, 11)]
        return np.column_stack([np.nanmean(x_along5[:, a:b], axis=1) for a, b in g])

    edges = np.linspace(0, m1, m5 + 1).astype(int)
    return np.column_stack([np.nanmean(x_along5[:, edges[i]:edges[i+1]], axis=1) for i in range(m5)])

def refine_dup_time(time_ms):
    tu = np.unique(time_ms)
    dt = np.diff(tu) / np.timedelta64(1, "s")
    small = dt[(dt > 0) & (dt < 60)]
    if small.size == 0:
        return time_ms
    dt_ms = int(round(float(np.median(small)) * 1000))

    t = time_ms.astype("datetime64[ms]").copy()
    out = t.copy()
    i, n = 0, len(t)
    while i < n:
        j = i + 1
        while j < n and t[j] == t[i]:
            j += 1
        k = j - i
        if k > 1:
            offs = (np.arange(k) - (k - 1) / 2.0) * (dt_ms / k)
            out[i:j] = t[i] + np.round(offs).astype(np.int64).astype("timedelta64[ms]")
        i = j
    return out

def hdf_to_one_nc(fn, out_dir):
    base = os.path.splitext(os.path.basename(fn))[0]
    out_nc = os.path.join(out_dir, f"{base}_ENA_cloud_1621.nc")

    hdf = SD(fn, SDC.READ)

    # Read 5 km geolocation and time
    lat = read_scaled_nan(hdf, "Latitude", np.float32)      # (406, 3)
    lon = read_scaled_nan(hdf, "Longitude", np.float32)     # (406, 3)
    lon = ((lon + 180) % 360) - 180  # Convert longitude to [-180, 180]

    tsec = read_scaled_nan(hdf, "Scan_Start_Time", np.float64)  # (406, 3)
    t1d = tsec[:, 0]
    good = np.isfinite(t1d) & (t1d >= 0)
    if good.sum() == 0:
        raise ValueError("no valid time")

    # Read 1 km cloud variables with original shape (2030, 11)
    re1   = read_scaled_nan(hdf, "Cloud_Effective_Radius_1621", np.float32)
    reu1  = read_scaled_nan(hdf, "Cloud_Effective_Radius_Uncertainty_1621", np.float32)
    cod1  = read_scaled_nan(hdf, "Cloud_Optical_Thickness_1621", np.float32)
    codu1 = read_scaled_nan(hdf, "Cloud_Optical_Thickness_Uncertainty_1621", np.float32)
    cwp1  = read_scaled_nan(hdf, "Cloud_Water_Path_1621", np.float32)
    cwpu1 = read_scaled_nan(hdf, "Cloud_Water_Path_Uncertainty_1621", np.float32)
    ctt1  = read_scaled_nan(hdf, "cloud_top_temperature_1km", np.float32)

    # Aggregate 1 km cloud variables to 5 km geolocation shape (406, 3)
    re5   = agg_1km_to_5km(re1,   lat.shape)
    reu5  = agg_1km_to_5km(reu1,  lat.shape)
    cod5  = agg_1km_to_5km(cod1,  lat.shape)
    codu5 = agg_1km_to_5km(codu1, lat.shape)
    cwp5  = agg_1km_to_5km(cwp1,  lat.shape)
    cwpu5 = agg_1km_to_5km(cwpu1, lat.shape)
    ctt5  = agg_1km_to_5km(ctt1,  lat.shape)

    # Keep only rows with valid scan time
    lat = lat[good]
    lon = lon[good]
    re5, reu5 = re5[good], reu5[good]
    cod5, codu5 = cod5[good], codu5[good]
    cwp5, cwpu5 = cwp5[good], cwpu5[good]
    ctt5 = ctt5[good]
    time = secs93_to_dt64ms(t1d[good])

    # Sort observations by time within each granule
    idx = np.argsort(time)
    time = time[idx]
    lat, lon = lat[idx], lon[idx]
    re5, reu5 = re5[idx], reu5[idx]
    cod5, codu5 = cod5[idx], codu5[idx]
    cwp5, cwpu5 = cwp5[idx], cwpu5[idx]
    ctt5 = ctt5[idx]

    # Slightly adjust duplicated scan times
    time_ref = refine_dup_time(time)

    # Flatten 2D scan/pixel arrays into 1D observation arrays
    nx = lat.shape[1]
    t_flat = np.repeat(time_ref, nx)
    lat_f = lat.reshape(-1)
    lon_f = lon.reshape(-1)

    re_f   = re5.reshape(-1)
    reu_f  = reu5.reshape(-1)
    cod_f  = cod5.reshape(-1)
    codu_f = codu5.reshape(-1)
    cwp_f  = cwp5.reshape(-1)
    cwpu_f = cwpu5.reshape(-1)
    ctt_f  = ctt5.reshape(-1)

    # Select observations inside the ENA 5° × 5° box
    in_box = (
        np.isfinite(lat_f) & np.isfinite(lon_f) &
        (lat_f >= ena_lat_min) & (lat_f <= ena_lat_max) &
        (lon_f >= ena_lon_min) & (lon_f <= ena_lon_max)
    )

    # Write an empty NetCDF file if no observations fall inside the ENA box
    if in_box.sum() == 0:
        ds = xr.Dataset(
            data_vars=dict(
                re_1621=("obs", np.array([], dtype=np.float32)),
                re_1621_unc=("obs", np.array([], dtype=np.float32)),
                COD_1621=("obs", np.array([], dtype=np.float32)),
                COD_1621_unc=("obs", np.array([], dtype=np.float32)),
                CWP_1621=("obs", np.array([], dtype=np.float32)),
                CWP_1621_unc=("obs", np.array([], dtype=np.float32)),
                CTT=("obs", np.array([], dtype=np.float32)),
            ),
            coords=dict(
                time=("obs", np.array([], dtype="datetime64[ms]")),
                lat=("obs", np.array([], dtype=np.float32)),
                lon=("obs", np.array([], dtype=np.float32)),
            ),
            attrs=dict(
                source_file=fn,
                source="MAC06S0 1km vars aggregated to 5km geolocation, then filtered to ENA box",
                box=f"lat[{ena_lat_min},{ena_lat_max}], lon[{ena_lon_min},{ena_lon_max}]",
                note="No points inside ENA box for this granule.",
            )
        )
        ds.to_netcdf(out_nc)
        return out_nc, 0

    ds = xr.Dataset(
        data_vars=dict(
            re_1621=("obs", re_f[in_box].astype(np.float32)),
            re_1621_unc=("obs", reu_f[in_box].astype(np.float32)),
            COD_1621=("obs", cod_f[in_box].astype(np.float32)),
            COD_1621_unc=("obs", codu_f[in_box].astype(np.float32)),
            CWP_1621=("obs", cwp_f[in_box].astype(np.float32)),
            CWP_1621_unc=("obs", cwpu_f[in_box].astype(np.float32)),
            CTT=("obs", ctt_f[in_box].astype(np.float32)),
        ),
        coords=dict(
            time=("obs", t_flat[in_box]),
            lat=("obs",  lat_f[in_box].astype(np.float32)),
            lon=("obs",  lon_f[in_box].astype(np.float32)),
        ),
        attrs=dict(
            source_file=fn,
            source="MAC06S0 1km vars aggregated to 5km geolocation, then filtered to ENA box",
            box=f"lat[{ena_lat_min},{ena_lat_max}], lon[{ena_lon_min},{ena_lon_max}]",
            note="Duplicate times spread using median small dt within this granule.",
        )
    )

    # Sort final observations by time
    ds = ds.isel(obs=np.argsort(ds.time.values))

    ds.to_netcdf(out_nc)
    return out_nc, int(ds.sizes["obs"])

# ===== run all files =====
files = sorted(glob.glob(f"{IN_DIR}/*.hdf"))
print("N files:", len(files))

kept = skipped = 0
for fn in files:
    try:
        out_nc, nobs = hdf_to_one_nc(fn, OUT_DIR)
        kept += 1
        print(f"[OK] {os.path.basename(fn)} -> {os.path.basename(out_nc)}  obs={nobs}")
    except Exception as e:
        skipped += 1
        print(f"[SKIP] {os.path.basename(fn)}  {repr(e)}")

print("kept:", kept, "skipped:", skipped)
print("OUT_DIR:", OUT_DIR)

In [ ]:
import xarray as xr
import glob

nc_files = sorted(glob.glob("/data/ggong/MODIS/ENA_nc_5x5/*_ENA_cloud*.nc"))

def _pre(ds):
    # Ensure time is datetime64 and keep only needed variables if necessary
    # ds["time"] = ds["time"].astype("datetime64[ns]")  # Usually not needed because time is already decoded
    return ds

ds = xr.open_mfdataset(
    nc_files,
    combine="nested",        # Key: do not use by_coords
    concat_dim="obs",        # Concatenate along the obs dimension
    preprocess=_pre,
    data_vars="minimal",
    coords="minimal",
    compat="override",
    join="outer",            # Conservative option
    parallel=False,          # Set to True if a dask environment is available
)

# Key: sort by time after concatenation
ds = ds.sortby("time")

print(ds)

In [ ]:
ds = ds.swap_dims({"obs": "time"})
ds = ds.sel(time=slice("2006-01-01", "2017-12-31"))
ds.to_netcdf('//data/ggong/MODIS/ENA_COD_merge_5x5.nc')

##### if you want to get the result for SGP, just replace any 'ENA' to 'SGP'